# Diagnóstico Bronze — steam_games_review

Tabla analizada: `steam_game_intelligence.bronze.steam_games_review`. Se comprueban nulos, claves repetidas y consistencia de los snapshots. El campo `reviews` es string según el esquema proporcionado; no se presupone su estructura interna.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y cobertura

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT app_id) AS distinct_app_ids,
  COUNT_IF(app_id IS NULL) AS null_app_id_rows,
  COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows,
  COUNT_IF(reviews IS NULL) AS null_reviews_rows,
  COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews_rows,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.steam_games_review;

## 2. Nulos por columna

In [ ]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(app_id IS NULL) AS app_id,
    COUNT_IF(name IS NULL) AS name,
    COUNT_IF(reviews IS NULL) AS reviews,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steam_games_review
) UNPIVOT (null_count FOR column_name IN (app_id, name, reviews, _ingested_at, _source))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Duplicados exactos por clave y día de ingesta

In [ ]:
%sql
SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date,
       COUNT(*) AS row_count,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT reviews) AS distinct_reviews
FROM steam_game_intelligence.bronze.steam_games_review
GROUP BY app_id, CAST(_ingested_at AS DATE)
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

## 4. Historial de snapshots por app_id

In [ ]:
%sql
SELECT app_id,
       COUNT(*) AS snapshot_rows,
       COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT reviews) AS distinct_reviews,
       MIN(_ingested_at) AS first_seen,
       MAX(_ingested_at) AS last_seen
FROM steam_game_intelligence.bronze.steam_games_review
WHERE app_id IS NOT NULL
GROUP BY app_id
HAVING COUNT(*) > 1
ORDER BY snapshot_rows DESC
LIMIT 100;

## 5. Irregularidades básicas

In [ ]:
%sql
SELECT
  COUNT_IF(app_id <= 0) AS invalid_app_id,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp,
  COUNT_IF(name IS NOT NULL AND TRIM(name) = '') AS blank_name,
  COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews,
  COUNT_IF(_source IS NOT NULL AND TRIM(_source) = '') AS blank_source
FROM steam_game_intelligence.bronze.steam_games_review;

## 6. Registros para inspección

In [ ]:
%sql
SELECT *
FROM steam_game_intelligence.bronze.steam_games_review
WHERE app_id <= 0
   OR name IS NULL OR TRIM(name) = ''
   OR reviews IS NULL OR TRIM(reviews) = ''
   OR _ingested_at IS NULL
LIMIT 200;

In [ ]:
%python
from datetime import datetime
import os

table_name = 'steam_game_intelligence.bronze.steam_games_review'
notebook_name = '02_diagnostico_steam_games_review'
queries = [('Consulta de celda SQL 1', "SELECT\n  COUNT(*) AS total_rows,\n  COUNT(DISTINCT app_id) AS distinct_app_ids,\n  COUNT_IF(app_id IS NULL) AS null_app_id_rows,\n  COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows,\n  COUNT_IF(reviews IS NULL) AS null_reviews_rows,\n  COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews_rows,\n  MIN(_ingested_at) AS first_ingested_at,\n  MAX(_ingested_at) AS last_ingested_at\nFROM steam_game_intelligence.bronze.steam_games_review;"), ('Consulta de celda SQL 2', 'SELECT column_name, null_count\nFROM (\n  SELECT\n    COUNT_IF(app_id IS NULL) AS app_id,\n    COUNT_IF(name IS NULL) AS name,\n    COUNT_IF(reviews IS NULL) AS reviews,\n    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,\n    COUNT_IF(_source IS NULL) AS _source\n  FROM steam_game_intelligence.bronze.steam_games_review\n) UNPIVOT (null_count FOR column_name IN (app_id, name, reviews, _ingested_at, _source))\nWHERE null_count > 0\nORDER BY null_count DESC;'), ('Consulta de celda SQL 3', 'SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date,\n       COUNT(*) AS row_count,\n       COUNT(DISTINCT name) AS distinct_names,\n       COUNT(DISTINCT reviews) AS distinct_reviews\nFROM steam_game_intelligence.bronze.steam_games_review\nGROUP BY app_id, CAST(_ingested_at AS DATE)\nHAVING COUNT(*) > 1\nORDER BY row_count DESC\nLIMIT 200;'), ('Consulta de celda SQL 4', 'SELECT app_id,\n       COUNT(*) AS snapshot_rows,\n       COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days,\n       COUNT(DISTINCT name) AS distinct_names,\n       COUNT(DISTINCT reviews) AS distinct_reviews,\n       MIN(_ingested_at) AS first_seen,\n       MAX(_ingested_at) AS last_seen\nFROM steam_game_intelligence.bronze.steam_games_review\nWHERE app_id IS NOT NULL\nGROUP BY app_id\nHAVING COUNT(*) > 1\nORDER BY snapshot_rows DESC\nLIMIT 100;'), ('Consulta de celda SQL 5', "SELECT\n  COUNT_IF(app_id <= 0) AS invalid_app_id,\n  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp,\n  COUNT_IF(name IS NOT NULL AND TRIM(name) = '') AS blank_name,\n  COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews,\n  COUNT_IF(_source IS NOT NULL AND TRIM(_source) = '') AS blank_source\nFROM steam_game_intelligence.bronze.steam_games_review;"), ('Consulta de celda SQL 6', "SELECT *\nFROM steam_game_intelligence.bronze.steam_games_review\nWHERE app_id <= 0\n   OR name IS NULL OR TRIM(name) = ''\n   OR reviews IS NULL OR TRIM(reviews) = ''\n   OR _ingested_at IS NULL\nLIMIT 200;")]

def markdown_value(value):
    if value is None:
        return "NULL"
    return str(value).replace("|", "\\|").replace("\r", " ").replace("\n", " ")

report = [
    f"# Resultados del diagn?stico ? {notebook_name}",
    "",
    f"- **Notebook:** `{notebook_name}`",
    f"- **Ejecutado:** `{datetime.now().astimezone().isoformat()}` (zona horaria del driver)",
    "- **Estado:** `completed`",
    "- **Objetivo:** conservar los resultados efectivos de las validaciones SQL del notebook para su revisi?n local.",
    "",
    "## Tablas inspeccionadas",
    f"- `{table_name}`",
    "",
    "## Consultas y resultados",
]
for title, query in queries:
    rows = [row.asDict(recursive=True) for row in spark.sql(query).limit(200).collect()]
    report.extend(["", f"### {title}", f"_Filas exportadas: {len(rows)}; m?ximo aplicado: 200._", ""])
    if not rows:
        report.append("No se detectaron registros que incumplan esta validaci?n o la consulta no devolvi? filas.")
        continue
    columns = list(rows[0].keys())
    report.append("| " + " | ".join(columns) + " |")
    report.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for row in rows:
        report.append("| " + " | ".join(markdown_value(row.get(column)) for column in columns) + " |")
report.extend(["", "## Hallazgos y conclusiones", "Los hallazgos deben interpretarse a partir de los conteos y muestras exportados; esta celda no inventa conclusiones ni marca como aprobada una validaci?n no ejecutada.", ""])

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/src/transformation/diagnosticos"
if marker not in context_path:
    raise RuntimeError(f"No se pudo determinar la ra?z del repositorio desde la ruta del notebook: {context_path}")
repo_path = context_path.split(marker, 1)[0]
workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
output_dir = os.path.join(workspace_repo_path, "execution_results")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"resultados_{notebook_name}.md")
temp_path = output_path + ".tmp"
with open(temp_path, "w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(report))
os.replace(temp_path, output_path)
print(f"Markdown generado: {output_path}; consultas exportadas: {len(queries)}")
